#### 0.Labels
We will setup the environment for our task2 in this step.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]
MATRIX = ROOT / "task1" / "data" / "matrix_5min"
OUT = ROOT / "task2" / "data"
assert MATRIX.exists(), MATRIX
sys.path.insert(0, "../code")
import config as C
import features as FT

In [2]:
from pyspark.sql import SparkSession, functions as F, Window

spark = C.get_spark("task2-labels")

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo
ORIGIN_EPOCH = int(datetime(2017, 1, 1, tzinfo=ZoneInfo("Asia/Singapore")).timestamp())

# Now read the parquet files
reader = spark.read.option("pathGlobFilter", "*.parquet")
wide = reader.parquet(str(MATRIX))
wide.show(5)

+------+----+---+---+----+----+----+----+----+----+----+----+----+----+---+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+---+----+---+---+---+----+---+---+---+---+---+----+----+---+----+---+---+---+---+----+---+---+---+---+---+---+---+---+---+----+---+---+---+----+----+-----------+
|  slot| S06|S07|S08|S100|S101|S102|S103|S104|S105|S106|S107|S108|S109|S11|S110|S111|S112|S113|S114|S115|S116|S117|S118|S119|S120|S121|S122|S123|S201|S202|S203|S204|S205|S207|S208|S209|S210|S211|S212|S213|S214|S215|S216|S217|S218|S219|S220|S221|S222|S223|S224|S226|S227|S228|S229|S230|S24|S24B|S29|S33|S35| S36|S40|S43|S44|S46|S50| S55| S60|S61| S63|S64|S66|S69|S71| S72|S77|S78|S79|S81|S82|S84|S88|S89|S90|S900|S91|S92|S94| S96| S97|source_year|
+------+----+---+---+----+----+----+----+----+----+----+----+----+----+---+----+----+----+----+----+----+-

In [4]:
import pandas as pd
stations_ids = pd.read_csv(MATRIX / "stations.csv").station_id.tolist()
long = wide.unpivot(
    ids="slot",
    values=stations_ids,
    variableColumnName="station_id",
    valueColumnName="value",
)
long.show(5)

+------+----------+-----+
|  slot|station_id|value|
+------+----------+-----+
|315360|       S06| NULL|
|315360|       S07|  0.0|
|315360|       S08|  0.0|
|315360|      S100|  0.0|
|315360|      S101| NULL|
+------+----------+-----+
only showing top 5 rows


In [5]:
# Here, we notice the potential existence of a certian type of anomaly, that 
# it may be the case that for a certain station within a month, it has zero wet 
# slots recorded, yet the expected number of that islandwise is far greater than
# zero, then it has little chance that this station stays within a region away
# from rain throughout the month. Rather, we mark them as anomalies and convert
# these zeros (no rain) to null.
long = (long
        .withColumn("ts", F.timestamp_seconds(F.lit(ORIGIN_EPOCH) + F.col("slot") * 300))
        .withColumn("ym", F.date_format("ts", "yyyy-MM")))

obs = (long.filter(F.col("value").isNotNull())
       .withColumn("wet", (F.col("value") > 0).cast("double")))

# At every slot, 5min window, find the average wet ratio
net = obs.groupBy("slot").agg(F.avg("wet").alias("p_net"))

# O represents the number of wet slots of a certain (station, month)
# E represents the expected number of wet slots, since that of each slot
# is a probability p_net in [0, 1], we can do E = Sum 1*p_net = Sum p_net 
sm = (obs.join(net, on="slot")
         .groupBy("station_id", "ym")
         .agg(F.count("*").alias("n"),
              F.sum("wet").alias("O"),
              F.sum("p_net").alias("E")))

stuck = sm.filter((F.col("O") == 0) & (F.col("E") >= 30)).orderBy("station_id", "ym")
stuck.show(20)

+----------+-------+----+---+------------------+
|station_id|     ym|   n|  O|                 E|
+----------+-------+----+---+------------------+
|      S226|2021-02|8043|0.0|31.608745730608916|
|      S226|2021-03|8928|0.0|234.81063195458876|
|      S24B|2017-01|8756|0.0|356.89587072527223|
|      S24B|2017-02|8003|0.0|187.99142089391043|
|      S24B|2017-03|8864|0.0|253.01861769225084|
|      S24B|2017-04|7025|0.0|227.87426936329774|
|       S36|2019-02|3922|0.0|31.505983229554545|
|       S36|2019-03|3680|0.0| 38.28045526721326|
|       S36|2019-04|3717|0.0| 136.9301782563144|
|       S36|2019-05|4372|0.0| 65.45890183455433|
|       S36|2019-06|2426|0.0| 37.14510903831073|
|       S36|2019-10|5073|0.0|135.42911181308523|
|       S36|2019-11|2928|0.0|102.78023278977777|
|       S36|2020-01|2203|0.0|31.557697001877663|
|       S36|2020-05|1389|0.0|32.007560727343666|
|       S36|2020-06|1920|0.0|115.93435630971476|
|       S36|2021-08|1158|0.0| 45.11219316114151|
|       S84|2019-12|

In [6]:
# Since the anomalies shown above all happened within the traning set and 
# has a size of 18, we can simply mark them as null observation using toPandas
stuck_pdf = stuck.toPandas()          
stuck_pdf.to_csv(OUT / "stuck_station_months.csv", index=False)

flags = spark.createDataFrame(stuck_pdf[["station_id", "ym"]]).withColumn("stuck", F.lit(True))

long_clean = (long
    .join(F.broadcast(flags), on=["station_id", "ym"], how="left")
    .withColumn("value", F.when(F.col("stuck"), F.lit(None)).otherwise(F.col("value")))
    .drop("stuck"))

# Two gauges also reported rain that never fell (S107 in August 2022, S36 in 2019-2021,
# see task1/docs/S107_gauge_fault_2022_08.txt); config.FAULTY lists the periods.
long_clean = FT.mask_faulty(long_clean)

In [7]:
long_clean.count() # row number shall remain the same

76579776

In [8]:
w = Window.partitionBy("station_id").orderBy("slot")
w_past = w.rowsBetween(-5, 0)
w_fut = w.rowsBetween(1, 6)

lab = (long_clean
       .withColumn("fut_sum", F.sum("value").over(w_fut))
       .withColumn("fut_n", F.count("value").over(w_fut))
       .withColumn("past_sum", F.sum("value").over(w_past))
       .withColumn("past_n", F.count("value").over(w_past)))

In [9]:
# Now label and construct some basic features, including timestamp, rainfall in past30 min.
valid = lab.filter((F.col("value").isNotNull()) & (F.col("fut_n") == 6))
valid = (valid
         .withColumn("y_mm", F.round("fut_sum", 1))
         .withColumn("past30_mm", F.when(F.col("past_n") == 6, F.round("past_sum", 1)))
         .withColumn("regime", F.when(F.col("past30_mm").isNull(), "unknown")
                                .when(F.col("past30_mm") > 0, "wet_now")
                                .otherwise("dry_now")))

valid = (valid
    .withColumn("year", F.year("ts"))
    .withColumn("year_end", F.year(F.col("ts") + F.expr("INTERVAL 30 MINUTES"))))

In [10]:
# Now discretize the label into buckets
from pyspark.ml.feature import Bucketizer
splits = [-float("inf"), 0.05, 0.25, 0.55, 1.05, 2.05, 5.05, 10.05, 20.05, float("inf")]
valid = Bucketizer(splits=splits, inputCol="y_mm", outputCol="label").transform(valid)

In [11]:
# Here we only select the rows whose next 30 min window still lies in the same year, namely
# we filter out those rows that spans over two years
labels = (valid
    .filter(F.col("year") == F.col("year_end"))                  
    .select("station_id", "slot", "ts", "year", "y_mm", "label", "past30_mm", "regime"))

labels.write.mode("overwrite").partitionBy("year").parquet(str(OUT / "labels"))

In [12]:
# The following is then the labeled dataset
labels = spark.read.parquet(str(OUT / "labels"))
labels.show(10)

+----------+------+-------------------+----+-----+---------+-------+----+
|station_id|  slot|                 ts|y_mm|label|past30_mm| regime|year|
+----------+------+-------------------+----+-----+---------+-------+----+
|      S108|420768|2021-01-01 00:00:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420769|2021-01-01 00:05:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420770|2021-01-01 00:10:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420771|2021-01-01 00:15:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420772|2021-01-01 00:20:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420773|2021-01-01 00:25:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420774|2021-01-01 00:30:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420775|2021-01-01 00:35:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420776|2021-01-01 00:40:00| 0.0|  0.0|      0.0|dry_now|2021|
|      S108|420777|2021-01-01 00:45:00| 0.0|  0.0|      0.0|dry_now|2021|
+----------+------+-------------------

#### Labels summary

Two cells. The first prints the tables to read: class balance (with the `y_mm` range inside each class), persistence by regime, wet share by year and split, and the diurnal and seasonal peaks. The second turns the checkpoints from lessons 3–5 into assertions, so a rerun that changes the labels stops here with an error instead of passing silently.

Expected: 47,511,766 samples; P(wet next 30 min) ≈ 0.630 when wet now and ≈ 0.024 when dry now; peak hour 15:00; highest wet frequency in January (rain amount peaks in November, see Task 1).

In [13]:
labels = spark.read.parquet(str(OUT / "labels")).cache()
total = labels.count()
wet = (F.col("label") > 0).cast("double")
print(f"samples: {total:,}")

by_label = (labels.groupBy("label")
            .agg(F.count("*").alias("n"), F.min("y_mm").alias("y_min"), F.max("y_mm").alias("y_max"))
            .withColumn("share", F.round(F.col("n") / total, 5))
            .orderBy("label"))
by_label.show()

labels.groupBy("regime").agg(F.count("*").alias("n"),
                             F.round(F.avg(wet), 4).alias("p_wet_next30")).orderBy("regime").show()

split = F.when(F.col("year") <= 2022, "1 train").when(F.col("year") == 2023, "2 val").otherwise("3 test")
labels.groupBy("year").agg(F.count("*").alias("n"), F.round(F.avg(wet), 4).alias("p_wet")).orderBy("year").show()
labels.groupBy(split.alias("split")).agg(F.count("*").alias("n"), F.round(F.avg(wet), 4).alias("p_wet")).orderBy("split").show()

labels.groupBy(F.hour("ts").alias("hour")).agg(F.round(F.avg(wet), 4).alias("p_wet")).orderBy(F.desc("p_wet")).show(3)
labels.groupBy(F.month("ts").alias("month")).agg(F.round(F.avg(wet), 4).alias("p_wet")).orderBy(F.desc("p_wet")).show(3)

samples: 47,511,766


+-----+--------+-----+-----+-------+
|label|       n|y_min|y_max|  share|
+-----+--------+-----+-----+-------+
|  0.0|44586115|  0.0|  0.0|0.93842|
|  1.0|  924336|  0.2|  0.2|0.01945|
|  2.0|  360170|  0.4|  0.4|0.00758|
|  3.0|  520926|  0.6|  1.0|0.01096|
|  4.0|  372605|  1.2|  2.0|0.00784|
|  5.0|  357875|  2.2|  5.0|0.00753|
|  6.0|  204046|  5.1| 10.0|0.00429|
|  7.0|  134367| 10.1| 20.0|0.00283|
|  8.0|   51326| 20.1| 71.8|0.00108|
+-----+--------+-----+-----+-------+



+-------+--------+------------+
| regime|       n|p_wet_next30|
+-------+--------+------------+
|dry_now|44320448|      0.0243|
|unknown|  285102|       0.062|
|wet_now| 2906216|      0.6296|
+-------+--------+------------+



+----+-------+------+
|year|      n| p_wet|
+----+-------+------+
|2017|5025679|0.0613|
|2018|4079470|0.0657|
|2019|4945897|0.0403|
|2020|6273733|0.0652|
|2021|6985964| 0.061|
|2022|7107273|0.0648|
|2023|6719274|0.0668|
|2024|6374476|0.0636|
+----+-------+------+



+-------+--------+------+
|  split|       n| p_wet|
+-------+--------+------+
|1 train|34418016|0.0602|
|  2 val| 6719274|0.0668|
| 3 test| 6374476|0.0636|
+-------+--------+------+



+----+------+
|hour| p_wet|
+----+------+
|  15|0.1167|
|  16|0.1131|
|  14|0.1086|
+----+------+
only showing top 3 rows


+-----+------+
|month| p_wet|
+-----+------+
|    1|0.0907|
|   11|0.0856|
|   12|0.0765|
+-----+------+
only showing top 3 rows


In [14]:
EXPECTED_TOTAL = 47_511_766
EXPECTED_LABELS = {0: 44_586_115, 1: 924_336, 2: 360_170, 3: 520_926, 4: 372_605,
                   5: 357_875, 6: 204_046, 7: 134_367, 8: 51_326}

assert total == EXPECTED_TOTAL, total
rows = by_label.collect()
assert {int(r.label): r.n for r in rows} == EXPECTED_LABELS

# every y_mm sits inside its own bucket
for r in rows:
    k = int(r.label)
    assert splits[k] <= r.y_min and r.y_max < splits[k + 1], r

# no missing values in the columns the model will use
key_cols = ["station_id", "slot", "ts", "year", "y_mm", "label", "regime"]
nulls = labels.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in key_cols]).first().asDict()
assert all(v == 0 for v in nulls.values()), nulls

# past30_mm is missing exactly when regime is unknown
assert labels.filter(F.col("past30_mm").isNull() != (F.col("regime") == "unknown")).count() == 0

# one row per (station, slot)
assert labels.select("station_id", "slot").distinct().count() == total

# stuck station-months are gone
stuck_keys = spark.createDataFrame(pd.read_csv(OUT / "stuck_station_months.csv")[["station_id", "ym"]])
leak = labels.withColumn("ym", F.date_format("ts", "yyyy-MM")).join(F.broadcast(stuck_keys), ["station_id", "ym"])
assert leak.count() == 0

# hand-checked example from lesson 3
r = labels.filter((F.col("station_id") == "S07") & (F.col("slot") == 660864)).first()
assert (r.y_mm, r.label, r.past30_mm, r.regime) == (3.2, 5.0, 0.0, "dry_now"), r

print("All label checks passed.")

All label checks passed.
